In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "quantum_jobs").is_dir()), cwd)

try:
    from quantum_jobs.db.paths import DB_PATH as PROJECT_DB_PATH

    db_path = Path(PROJECT_DB_PATH).resolve()
except Exception:
    db_path = (repo_root / "quantum_jobs.db").resolve()

print(f"Using database: {db_path}")

if not db_path.exists():
    raise FileNotFoundError(
        f"Database not found at {db_path}. "
        "Expected the project database at repo root; refusing to create a blank SQLite file."
    )

conn = sqlite3.connect(f"file:{db_path}?mode=rw", uri=True)


In [ ]:
def run_query(query: str):
    return pd.read_sql(query, conn)


In [ ]:
query = """
SELECT
    company,
    COALESCE(pulled_date, substr(pulled_at, 1, 10)) AS date,
    COUNT(*) AS job_count
FROM job_snapshots
GROUP BY company, date
ORDER BY date, company
"""

df = run_query(query)
df.head()


In [ ]:
plt.figure(figsize=(10, 5))
for company, group in df.groupby("company"):
    plt.plot(group["date"], group["job_count"], marker="o", label=company)

plt.title("Jobs Over Time by Company")
plt.xlabel("Date")
plt.ylabel("Job Count")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()
